# Simulação de respostas — Pesquisa Cesop 04829

**Tema:** *Percepção dos brasileiros sobre o racismo no Brasil* (microdado com **144 variáveis**; base empírica com 2000 entrevistas).

Este notebook é a parte de **simulação de opinião pública com modelo de linguagem** do projeto da disciplina: gera **N** questionários sintéticos, envia o instrumento em **blocos** ao modelo (para respeitar o limite de contexto) e grava respostas nos **mesmos códigos numéricos** do arquivo original `04829.SAV`, com checagem de domínio.

**Modos de execução**
- **`USE_MOCK_LLM = True`:** percorre todo o pipeline (dados, blocos, validação) **sem** carregar rede neural — útil para testar caminhos e lógica.
- **`USE_MOCK_LLM = False`:** usa o modelo aberto **Qwen2.5-3B-Instruct** em **4-bit** via Hugging Face (recomenda-se **GPU** no Colab). Se a resposta não for JSON válido, o código tenta de novo; opcionalmente pode recair no mock após várias falhas (registre isso no relatório se ocorrer).

**Arquivos gerados:** tabela sintética (CSV), log detalhado (JSON) e **`llm_run_metadata_*.json`** com hiperparâmetros para documentação.

**Teste rápido:** `RUN_SMOKE_TEST = True` produz **1** respondente e **1** bloco após o perfil sócio; nomes de arquivo com `_SMOKE_` para não misturar com a rodada completa.

## Ordem de execução (de cima para baixo)

1. Instalar dependências (`%pip`).
2. **Config:** `drive.mount`, pastas e `USE_MOCK_LLM` / `RUN_SMOKE_TEST`.
3. **Carregar dados** (assert com mensagem útil se faltar arquivo).
4. Funções de prompt e validação.
5. Modelo LLM / mock.
6. **Rodar simulação** (última célula de código — pode demorar).

---

## Dados no Google Drive

1. Use a pasta **`DadosModeloLinguagens`** no Meu Drive com **estes** arquivos:

| Arquivo | Função |
|---------|--------|
| `04829.SAV` | Microdado SPSS |
| `survey_dictionary.json` | Dicionário de variáveis e blocos (pode vir do repositório: `tools/export_survey_dictionary.py`) |

2. **Links de referência** (se precisar baixar de novo e enviar ao Drive): [`.SAV`](https://drive.google.com/file/d/1fJiR7wx6R3AACKdMLLpOBy1RjY_o2Sng/view?usp=drive_link) · [`survey_dictionary.json`](https://drive.google.com/file/d/1ijLzS-LmegfYrXd0ge5URJD-iQXfj7Rj/view?usp=drive_link)

3. Na célula seguinte: `drive.mount('/content/drive')`. O caminho da pasta **DadosModeloLinguagens** (Meu Drive → essa pasta) no Colab é:

`/content/drive/MyDrive/DadosModeloLinguagens`

*(Na interface em português aparece “Meu Drive”; no disco montado o nome da pasta é sempre **`MyDrive`**, sem acento.)*

**Saídas:** subpasta **`saida_simulacao`** dentro da mesma pasta do Drive (CSV, logs, metadata).

In [5]:
%pip install -q pyreadstat pandas tqdm transformers accelerate safetensors bitsandbytes

In [6]:
from __future__ import annotations

import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

# --- Google Drive ---
from google.colab import drive

drive.mount('/content/drive')

# Pasta no Drive com 04829.SAV e survey_dictionary.json
# "Meu Drive" na interface = MyDrive no caminho
DRIVE_SURVEY_DIR = Path("/content/drive/MyDrive/DadosModeloLinguagens")

OUT_DIR = DRIVE_SURVEY_DIR / "saida_simulacao"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAV_PATH = DRIVE_SURVEY_DIR / "04829.SAV"
DICT_PATH = DRIVE_SURVEY_DIR / "survey_dictionary.json"

N_RESPONDENTES = 200
RNG_SEED = 42
USE_BOOTSTRAP_SOCIO = True

# --- LLM ---
USE_MOCK_LLM = False  # False = carrega Qwen em 4-bit (exige GPU T4 ou superior)
LLM_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
LLM_TEMPERATURE = 0.25
LLM_MAX_NEW_TOKENS = 4096
LLM_N_RETRIES = 3
LLM_MAX_INPUT_TOKENS = 8192
LLM_FALLBACK_MOCK_ON_FAILURE = True
HF_TOKEN = None

RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    OUT_CSV = OUT_DIR / f"sintetico_04829_SMOKE_seed{RNG_SEED}.csv"
    LOG_PATH = OUT_DIR / f"sintetico_04829_log_SMOKE_seed{RNG_SEED}.json"
    LLM_METADATA_PATH = OUT_DIR / f"llm_run_metadata_SMOKE_seed{RNG_SEED}.json"
else:
    OUT_CSV = OUT_DIR / f"sintetico_04829_N{N_RESPONDENTES}_seed{RNG_SEED}.csv"
    LOG_PATH = OUT_DIR / f"sintetico_04829_log_N{N_RESPONDENTES}_seed{RNG_SEED}.json"
    LLM_METADATA_PATH = OUT_DIR / f"llm_run_metadata_N{N_RESPONDENTES}_seed{RNG_SEED}.json"

print("Dados: ", DRIVE_SURVEY_DIR)
print("Saídas:", OUT_DIR)
print("USE_MOCK_LLM:", USE_MOCK_LLM)


Mounted at /content/drive
Dados:  /content/drive/MyDrive/DadosModeloLinguagens
Saídas: /content/drive/MyDrive/DadosModeloLinguagens/saida_simulacao
USE_MOCK_LLM: False


In [7]:
# Carrega microdado e dicionário
if not SAV_PATH.is_file() or not DICT_PATH.is_file():
    faltando = [str(x) for x in (SAV_PATH, DICT_PATH) if not x.is_file()]
    try:
        itens = (
            "\n".join(f"  • {p.name}" for p in sorted(DRIVE_SURVEY_DIR.iterdir()))
            if DRIVE_SURVEY_DIR.is_dir()
            else "  (pasta inexistente)"
        )
    except OSError:
        itens = "  (não foi possível listar)"
    raise FileNotFoundError(
        "Arquivo(s) não encontrado(s):\n"
        + "\n".join(faltando)
        + f"\n\nPasta: {DRIVE_SURVEY_DIR}\n"
        "Conteúdo atual:\n"
        + itens
    )

df_emp, meta = pyreadstat.read_sav(str(SAV_PATH))
with open(DICT_PATH, "r", encoding="utf-8") as f:
    survey_doc = json.load(f)

blocks = survey_doc["suggested_llm_blocks"]
var_defs = {v["name"]: v for v in survey_doc["variables"]}
all_cols = list(df_emp.columns)
assert len(all_cols) == 144

SOCIO_VARS = next(b["variables"] for b in blocks if b["id"] == "socio_demografia")

print("Empírico:", df_emp.shape)
print("Blocos LLM:", len(blocks), "| vars:", sum(len(b["variables"]) for b in blocks))
print("Variáveis sócio (bootstrap):", len(SOCIO_VARS))


Empírico: (2000, 144)
Blocos LLM: 12 | vars: 144
Variáveis sócio (bootstrap): 20


In [8]:
def _fmt_label(v: dict) -> str:
    lab = v.get("label") or ""
    return lab if isinstance(lab, str) else ""


def build_block_prompt(
    block_id: str,
    variable_names: list[str],
    var_defs: dict,
    context: dict,
) -> str:
    """Monta o texto enviado ao modelo: instruções + contexto + itens do bloco."""
    lines = [
        "Instrução: simule um entrevistado que responde ao questionário da pesquisa (Cesop), de forma coerente com o perfil e com as respostas já dadas abaixo.",
        "Saída obrigatória: um único objeto JSON, sem texto antes ou depois, sem cercas de código markdown.",
        "Cada chave do JSON deve ser o nome da variável indicado em cada item; os valores são os códigos numéricos das opções.",
        "Use null apenas quando não resposta ou 'não se aplica' fizer sentido para aquele item.",
        f"Bloco: {block_id}",
        "",
        "--- Perfil e respostas já preenchidas (mantenha coerência): ---",
        json.dumps(context, ensure_ascii=False),
        "",
        "--- Itens deste bloco (responda todos no JSON) ---",
    ]
    for name in variable_names:
        v = var_defs[name]
        lines.append(f"Variável: {name}")
        desc = _fmt_label(v).strip() or name
        lines.append(f"Enunciado: {desc}")
        vl = v.get("value_labels")
        if vl:
            lines.append("Opções (código: texto):")
            for code, text in vl.items():
                lines.append(f"  {code}: {text}")
        else:
            lines.append(
                "(Sem rótulos no dicionário — use número coerente com o tipo da variável no questionário.)"
            )
        lines.append("")
    lines.append("Retorne somente o JSON com as chaves: " + ", ".join(variable_names))
    return "\n".join(lines)


def _coerce_code(vl: dict | None, raw) -> float | None:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    if vl is None:
        try:
            x = float(raw)
        except (TypeError, ValueError):
            return None
        return x
    key = str(int(raw)) if isinstance(raw, (int, float)) and float(raw).is_integer() else str(raw)
    if key not in vl:
        try:
            key = str(int(float(raw)))
        except (TypeError, ValueError):
            return None
    return float(key) if key in vl else None


def validate_and_merge(
    block_vars: list[str],
    parsed: dict,
    var_defs: dict,
    log: list,
) -> dict:
    out = {}
    for name in block_vars:
        if name not in parsed:
            log.append({"var": name, "error": "missing_key"})
            out[name] = np.nan
            continue
        vdef = var_defs[name]
        vl = vdef.get("value_labels")
        val = _coerce_code(vl, parsed[name])
        if vl is not None and val is None and parsed[name] is not None:
            log.append({"var": name, "error": "invalid_code", "got": parsed[name]})
        out[name] = np.nan if val is None else val
    return out


def extract_json_object(text: str) -> dict:
    """Extrai o primeiro objeto JSON válido (tolerante a ```json fences)."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\s*", "", text)
        text = re.sub(r"\s*```$", "", text).strip()
    dec = json.JSONDecoder()
    for i, ch in enumerate(text):
        if ch == "{":
            try:
                obj, _ = dec.raw_decode(text[i:])
                return obj
            except json.JSONDecodeError:
                continue
    raise ValueError("Nenhum JSON válido encontrado na resposta")

In [9]:
import os
import time

rng = np.random.default_rng(RNG_SEED)

_llm_model = None
_llm_tokenizer = None

def mock_llm_json_block(
    block_id: str,
    variable_names: list[str],
    var_defs: dict,
    df_emp: pd.DataFrame,
    context: dict,
) -> dict:
    """Baseline: códigos válidos amostrados do empírico ou dos rótulos (sem LLM)."""
    out = {}
    for name in variable_names:
        vdef = var_defs[name]
        vl = vdef.get("value_labels")
        s = df_emp[name].dropna()
        if len(s) == 0:
            out[name] = None
            continue
        if vl is not None:
            codes = [float(k) for k in vl.keys()]
            pick = float(rng.choice(codes))
        else:
            pick = float(s.sample(1, random_state=int(rng.integers(1 << 30))).iloc[0])
        out[name] = pick
    return out


def _load_hf_llm():
    """Carrega uma vez o modelo em 4-bit (BitsAndBytes) — adequado a GPU T4 no Colab."""
    global _llm_model, _llm_tokenizer
    if _llm_model is not None:
        return
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if not torch.cuda.is_available():
        raise RuntimeError(
            "GPU não disponível. Use USE_MOCK_LLM=True ou habilite GPU: Runtime > "
            "Change runtime type > GPU no Colab."
        )

    tok = HF_TOKEN or os.environ.get("HF_TOKEN")
    kwargs = {"trust_remote_code": True}
    if tok:
        kwargs["token"] = tok

    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    t0 = time.perf_counter()
    _llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, **kwargs)
    _llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_ID,
        quantization_config=quant,
        device_map="auto",
        **kwargs,
    )
    print(f"Modelo carregado em {time.perf_counter() - t0:.1f}s | {LLM_MODEL_ID}")


def _llm_generate_raw(system_prompt: str, user_prompt: str) -> str:
    import torch

    _load_hf_llm()
    model, tokenizer = _llm_model, _llm_tokenizer
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    if tokenizer.chat_template is None:
        prompt = f"{system_prompt}\n\n###\n\n{user_prompt}\n\n### Resposta JSON:\n"
    else:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=LLM_MAX_INPUT_TOKENS,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=min(LLM_MAX_NEW_TOKENS, 8192),
            do_sample=True,
            temperature=LLM_TEMPERATURE,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(gen, skip_special_tokens=True)


def call_llm_json(
    system_prompt: str,
    user_prompt: str,
    block_id: str,
    variable_names: list[str],
    var_defs: dict,
    df_emp: pd.DataFrame,
    context: dict,
):
    """Mock, LLM real com retries, ou fallback para mock conforme config."""
    if USE_MOCK_LLM:
        return mock_llm_json_block(
            block_id, variable_names, var_defs, df_emp, context
        )

    keys_hint = ", ".join(variable_names)
    repair = ""
    last_err = None

    for attempt in range(LLM_N_RETRIES):
        up = user_prompt + repair
        try:
            text = _llm_generate_raw(system_prompt, up)
            # Validate if the generated text is a valid JSON object
            extract_json_object(text)
            return text
        except Exception as e:
            last_err = e
            repair = (
                f"\n\n### Correção obrigatória (tentativa {attempt + 2}/{LLM_N_RETRIES})\n"
                "Responda somente um único objeto JSON válido (sem markdown, sem comentários). "
                f"Chaves obrigatórias exatamente: {keys_hint}. Erro ao interpretar a resposta anterior: {e!s}"
            )

    # If all LLM attempts fail, check for fallback
    if LLM_FALLBACK_MOCK_ON_FAILURE:
        print(
            f"[aviso] Bloco {block_id}: fallback para mock após {LLM_N_RETRIES} falhas. Último erro: {last_err}"
        )
        return mock_llm_json_block(
            block_id, variable_names, var_defs, df_emp, context
        )
    raise RuntimeError(f"Todas as tentativas do LLM falharam e fallback para mock está desativado. Último erro: {last_err}")


SYSTEM_PROMPT = (
    "Você preenche questionários de pesquisa de opinião com códigos fechados. "
    "A saída deve ser somente um objeto JSON válido, sem explicações, sem markdown."
)


## Teste rápido (recomendado antes da rodada longa)

Com **`RUN_SMOKE_TEST = True`** na configuração, o notebook gera **um** perfil sintético e executa **uma** inferência no primeiro bloco de perguntas após o bloco sócio (em geral `P1`). Os arquivos são salvos com sufixo **`_SMOKE_`**.

Para o resultado final do projeto, use **`RUN_SMOKE_TEST = False`** e `N_RESPONDENTES` conforme o enunciado (≥ 200, alinhado com o grupo).

In [ ]:
from datetime import date

def _row_to_jsonable(row: pd.Series, keys: list[str]) -> dict:
    out = {}
    for k in keys:
        v = row[k]
        if pd.isna(v):
            out[k] = None
        elif isinstance(v, (np.integer, int)):
            out[k] = int(v)
        elif isinstance(v, (np.floating, float)):
            out[k] = None if np.isnan(v) else float(v)
        elif isinstance(v, pd.Timestamp):
            out[k] = v.isoformat()
        elif isinstance(v, date):
            out[k] = v.isoformat()
        else:
            out[k] = v
    return out


def run_simulation(
    n_respondentes: int | None = None,
    max_llm_blocks_per_respondent: int | None = None,
) -> tuple[pd.DataFrame, list]:
    """Simulação completa ou parcial.

    - `max_llm_blocks_per_respondent`: se definido, interrompe após N chamadas ao LLM
      (blocos sócio ignorados antes da contagem). Útil para o teste rápido (`RUN_SMOKE_TEST`).
    """
    n_resp = int(n_respondentes if n_respondentes is not None else N_RESPONDENTES)
    rows = []
    global_logs = []

    for resp_idx in tqdm(range(n_resp), desc="Respondentes"):
        seed_row = df_emp.sample(1, random_state=int(rng.integers(1 << 31))).iloc[0]
        if USE_BOOTSTRAP_SOCIO:
            # Perfil empírico; demais blocos vêm do LLM (ou mock)
            context = _row_to_jsonable(seed_row, SOCIO_VARS)
        else:
            context = {}

        row_log = {"respondent": resp_idx, "blocks": []}
        llm_blocks_done = 0

        for blk in blocks:
            bid = blk["id"]
            bvars = blk["variables"]
            if USE_BOOTSTRAP_SOCIO and bid == "socio_demografia":
                row_log["blocks"].append(
                    {"block": bid, "skipped_llm": True, "issues": []}
                )
                continue

            if (
                max_llm_blocks_per_respondent is not None
                and llm_blocks_done >= max_llm_blocks_per_respondent
            ):
                row_log["blocks"].append(
                    {"block": bid, "skipped": True, "reason": "max_llm_blocks_reached"}
                )
                break

            user_prompt = build_block_prompt(bid, bvars, var_defs, context)
            try:
                raw = call_llm_json(
                    SYSTEM_PROMPT,
                    user_prompt,
                    bid,
                    bvars,
                    var_defs,
                    df_emp,
                    context,
                )
                if isinstance(raw, str):
                    parsed = extract_json_object(raw)
                else:
                    parsed = raw
            except Exception as e:
                row_log["blocks"].append({"block": bid, "error": str(e)})
                parsed = {}

            blk_log = []
            merged = validate_and_merge(bvars, parsed, var_defs, blk_log)
            context.update(merged)
            row_log["blocks"].append({"block": bid, "issues": blk_log})
            llm_blocks_done += 1

        final_row = {k: context.get(k, np.nan) for k in all_cols}
        row_log["final_row"] = final_row
        global_logs.append(row_log)
        rows.append(final_row)

    out_df = pd.DataFrame(rows)
    out_df = out_df[all_cols]
    return out_df, global_logs


if RUN_SMOKE_TEST:
    synth_df, logs = run_simulation(
        n_respondentes=1,
        max_llm_blocks_per_respondent=1,
    )
else:
    synth_df, logs = run_simulation()

synth_df.to_csv(OUT_CSV, index=False)
with open(LOG_PATH, "w", encoding="utf-8") as f:
    json.dump(logs, f, ensure_ascii=False, indent=2)

n_llm_blocks = sum(
    1
    for b in blocks
    if not (USE_BOOTSTRAP_SOCIO and b["id"] == "socio_demografia")
)
n_eff = 1 if RUN_SMOKE_TEST else N_RESPONDENTES
total_llm_calls = 1 if RUN_SMOKE_TEST else (N_RESPONDENTES * n_llm_blocks)
meta_doc = {
    "RUN_SMOKE_TEST": RUN_SMOKE_TEST,
    "USE_MOCK_LLM": USE_MOCK_LLM,
    "LLM_MODEL_ID": None if USE_MOCK_LLM else LLM_MODEL_ID,
    "quantization": None if USE_MOCK_LLM else "4-bit (BitsAndBytes nf4, bf16 compute)",
    "LLM_TEMPERATURE": LLM_TEMPERATURE,
    "LLM_MAX_NEW_TOKENS": LLM_MAX_NEW_TOKENS,
    "LLM_N_RETRIES": LLM_N_RETRIES,
    "LLM_MAX_INPUT_TOKENS": LLM_MAX_INPUT_TOKENS,
    "USE_BOOTSTRAP_SOCIO": USE_BOOTSTRAP_SOCIO,
    "N_RESPONDENTES_config": N_RESPONDENTES,
    "N_RESPONDENTES_efetivo": n_eff,
    "RNG_SEED": RNG_SEED,
    "n_blocks_por_respondente_llm": n_llm_blocks,
    "total_chamadas_llm_aprox": total_llm_calls,
    "smoke_max_llm_blocks_per_respondent": 1 if RUN_SMOKE_TEST else None,
}
with open(LLM_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(meta_doc, f, ensure_ascii=False, indent=2)

print("Salvo:", OUT_CSV)
print("Log:", LOG_PATH)
print("Metadata LLM:", LLM_METADATA_PATH)
print(synth_df.shape)
synth_df.head(2)


Respondentes:   0%|          | 0/200 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo carregado em 95.1s | Qwen/Qwen2.5-3B-Instruct
